# 빠알리어 TTS 생성기 (Facebook MMS 산스크리트어)

SuttaLog2 앱용 고품질 빠알리어 음성 파일 생성

**사용법**: 런타임 → 모두 실행 → 자동 완료 (약 15~25분)

텍스트 목록은 GitHub에서 자동 다운로드됩니다.

In [ ]:
# 1단계: 패키지 설치 + GPU 확인
!pip install -q transformers torch torchaudio soundfile pydub
!apt-get -qq install ffmpeg
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print('설치 완료!')

In [ ]:
# 2단계: Facebook MMS 산스크리트어 TTS 모델 로드 (인증 불필요)
import torch
from transformers import VitsModel, AutoTokenizer
import soundfile as sf
import os, re, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'facebook/mms-tts-san'
model = VitsModel.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
sample_rate = model.config.sampling_rate
print(f'Device: {device} | Sample rate: {sample_rate} | 모델 로드 완료!')

In [ ]:
# 3단계: 빠알리어 로마자 → 데바나가리 변환 함수
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = {
    '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907',
    '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913',
}
VOWELS_DEP = {
    '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f',
    '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b',
}
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''
    i = 0
    s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t':
            result += ch; i += 1; continue
        if ch == '\u1e43':
            result += '\u0902'; i += 1; continue
        three = s[i:i+3]
        two = s[i:i+2]
        consonant = None
        consumed = 0
        if three in CONSONANTS:
            consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS:
            consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS:
            consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND:
                result += consonant + VOWELS_DEP[s[i]]; i += 1
            else:
                result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND:
            result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

# 변환 테스트 (5개)
tests = ['buddhaṃ', 'dhammaṃ', 'saraṇaṃ gacchāmi', 'nibbāna', 'ṭhāna']
for t in tests:
    print(f'  {t} → {pali_to_devanagari(t)}')
print('변환 함수 준비 완료!')

In [ ]:
# 4단계: GitHub에서 텍스트 목록 다운로드 (업로드 불필요)
import json, urllib.request

GITHUB_RAW = 'https://raw.githubusercontent.com/ReachToWisdom/SuttaLog2/main'

# 테스트: 28개 / 전체: 528개 — 아래 파일명만 바꾸면 전환
TEXT_FILE = 'tts-texts-test28.json'   # 테스트용 28개
# TEXT_FILE = 'tts-texts.json'        # 전체 528개

url = f'{GITHUB_RAW}/{TEXT_FILE}'
print(f'다운로드: {url}')
response = urllib.request.urlopen(url)
texts = json.loads(response.read().decode('utf-8'))
print(f'{len(texts)}개 텍스트 로드 완료')

In [ ]:
# 5단계: 음성 생성 (528개, 약 15~25분)
import os, re, json
import soundfile as sf
import torch

os.makedirs('audio', exist_ok=True)

def text_to_filename(text):
    name = text.lower().strip()
    name = re.sub(r'[\n\r]', ' ', name)
    name = re.sub(r'[^a-z\u0101\u012b\u016b\u1e6d\u1e0d\u1e47\u1e45\u00f1\u1e43\u1e37 0-9\s\-]', '', name)
    name = re.sub(r'\s+', '-', name.strip())
    return name[:80]

total = len(texts)
errors = []
generated = 0

for i, text in enumerate(texts):
    fname = text_to_filename(text)
    outpath = f'audio/{fname}.wav'
    if os.path.exists(outpath):
        print(f'[{i+1}/{total}] skip: {fname}')
        continue
    try:
        clean_text = text.replace('\n', ' ').strip()
        devanagari = pali_to_devanagari(clean_text)
        inputs = tokenizer(devanagari, return_tensors='pt').to(device)
        with torch.no_grad():
            output = model(**inputs).waveform
        audio = output.squeeze().cpu().numpy()
        sf.write(outpath, audio, samplerate=sample_rate)
        generated += 1
        duration = len(audio) / sample_rate
        print(f'[{i+1}/{total}] {fname} ({duration:.1f}s)')
    except Exception as e:
        errors.append((text, str(e)))
        print(f'[{i+1}/{total}] ERR: {fname} - {e}')

print(f'\n완료! 신규 생성: {generated}, 오류: {len(errors)}, 스킵: {total - generated - len(errors)}')
if errors:
    print('오류 목록:')
    for t, e in errors:
        print(f'  {t[:50]} → {e}')

In [ ]:
# 6단계: MP3 변환 + ZIP 다운로드
import os, json, glob, zipfile
from pydub import AudioSegment
from google.colab import files

os.makedirs('audio_mp3', exist_ok=True)
wav_files = glob.glob('audio/*.wav')
for wf in wav_files:
    mp3f = wf.replace('audio/', 'audio_mp3/').replace('.wav', '.mp3')
    if not os.path.exists(mp3f):
        AudioSegment.from_wav(wf).export(mp3f, format='mp3', bitrate='64k')
print(f'MP3 변환: {len(wav_files)}개')

mapping = {}
for text in texts:
    fname = text_to_filename(text)
    if os.path.exists(f'audio_mp3/{fname}.mp3'):
        mapping[text] = f'{fname}.mp3'

with open('audio_mp3/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile('pali-audio.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('audio_mp3'):
        for fn in fnames:
            zf.write(os.path.join(root, fn), fn)

size_mb = os.path.getsize('pali-audio.zip') / (1024*1024)
print(f'\npali-audio.zip ({size_mb:.1f} MB) - {len(mapping)}개 파일')
print('다운로드 시작...')
files.download('pali-audio.zip')